# ส่วนที่ 3 · Extractive: Lead-3 Baseline + วิเคราะห์ Extractive Bias ใน ThaiSum

**ผู้รับผิดชอบ:** คนที่ 3 (คู่ B) · **ส่ง:** ตาราง ROUGE (Lead-1, 2, 3) + สถิติ Novel n-grams + `predictions_lead3.csv`

**วิธีใช้:** Runtime → **Run all** (ไม่ต้องใช้ GPU)

### วัตถุประสงค์และการตอบข้อสงสัยของทีม:
1. **สร้าง Baseline Lead-3:** ดึง 3 ประโยคแรกของเนื้อข่าวมาต่อเป็นบทสรุป เพื่อเป็นเส้นฐานเปรียบเทียบกับ TextRank (คนที่ 2), mT5 (คนที่ 4), และ mBART (คนที่ 5)
2. **ทดสอบบน Test Set เดียวกัน:** ใช้ชุดทดสอบเดียวกับทุกคนในโปรเจกต์ (500 ข่าวแรกตาม `02-textrank.ipynb` และ `04-mt5.ipynb` หรือ `data/processed/test.jsonl`)
3. **พิสูจน์สมมติฐาน "ThaiSum ลอกต้นข่าวจน ROUGE-2 สูง":**
   - วัด **Novel n-grams** (% คำและคู่คำใน Reference Summary ที่ไม่อยู่ในเนื้อข่าว)
   - วัด **Lead-3 Overlap** (สัดส่วนของคำคู่ใน Reference Summary ที่ตรงกับ 3 ประโยคแรก)

In [1]:
!pip -q install datasets pythainlp rouge_score scikit-learn python-crfsuite pandas


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. ตั้งค่า (แก้แค่ช่องนี้)

In [2]:
# ข้อมูล: None = โหลด ThaiSum จาก Hugging Face · ใส่ path = ใช้ไฟล์ของคนที่ 1
TRAIN_CSV, VAL_CSV, TEST_CSV = None, None, None
import os
TEST_JSONL = next((p for p in ["../data/processed/test.jsonl", "data/processed/test.jsonl"] if os.path.exists(p)), None)  # test กลางของคนที่ 1 (None = โหลดจาก HF)
VAL_SIZE   = 300
TEST_SIZE  = 500                    # ทุกโมเดลต้องใช้ test ชุดเดียวกัน

SENT_ENGINE = "crfcut"              # ตัดประโยค: "crfcut" (โมเดล CRF ของ PyThaiNLP) หรือ "space"
MIN_LEN     = 15                    # ประโยคสั้นกว่านี้ (ตัวอักษร) จะถูกรวมกับประโยคถัดไป
K_LIST      = [1, 2, 3, 4]          # จำนวนประโยคแรกที่นำมาเป็นบทสรุป
OUTPUT_NAME = "lead3"               # ชื่อไฟล์ผลลัพธ์
SEED        = 42

## 2. โหลดข้อมูล (ชุด Test เดียวกับทุกคน)

In [3]:
import pandas as pd
from datasets import load_dataset

def from_thaisum(split, n, shuffle=False):
    ds = load_dataset("pythainlp/thaisum", split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)
    return pd.DataFrame(list(ds.take(n)))[["body", "summary"]]

def load_data(file_path, split, n, shuffle=False):
    if file_path and file_path.endswith(".jsonl"):
        df = pd.read_json(file_path, lines=True).head(n)   # ใช้ n ข่าวแรก ให้ตรงกับทุกโมเดล
    elif file_path:
        df = pd.read_csv(file_path).head(n)
    else:
        df = from_thaisum(split, n, shuffle)
    df = df.dropna(subset=["body", "summary"])
    return df[(df.body.str.len() > 0) & (df.summary.str.len() > 0)].reset_index(drop=True)

val_df  = load_data(VAL_CSV, "validation", VAL_SIZE)
test_df = load_data(TEST_JSONL or TEST_CSV, "test", TEST_SIZE)
print(f"val: {len(val_df):,} ข่าว | test: {len(test_df):,} ข่าว")
print("\nตัวอย่างข่าว:", test_df.body[0][:200], "...\nบทสรุปจริง:", test_df.summary[0])

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: 300 ข่าว | test: 500 ข่าว

ตัวอย่างข่าว: ถูกอกถูกใจชาวโซเชียลไปจำนวนมาก สำหรับคลิปวิดีโอไวรัลสุดขำขัน จากผู้ใช้เฟซบุ๊กชื่อว่า ของ ใคร เป็นคลิปสาวประเภทสองคนหนึ่ง นั่งถือสาก รายงานข่าวอยู่บนโครงป้ายสีเขียวที่ปักอยู่กลางหนองน้ำ พร้อมกับพูดรายง ...
บทสรุปจริง: โลกโซเชียลแห่แชร์คลิปวิดีโอไวรัลสุดขำขัน ขณะสาวประเภทสองคนหนึ่งนั่งรายงานข่าวอยู่บนโครงป้าย ก่อนที่สุดท้ายจะหงายหลังตกลงน้ำ


## 3. ROUGE สำหรับภาษาไทย (กติกากลาง: ตัดคำ newmm ก่อน)

In [4]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {
        k: round(100 * sum(s[k].fmeasure for s in scores) / len(scores), 2)
        for k in ["rouge1", "rouge2", "rougeL"]
    }

# ทดสอบตัวอย่าง
print("ทดสอบ ROUGE:", thai_rouge(["นายกรัฐมนตรีประชุมครม.วันนี้"], ["วันนี้นายกรัฐมนตรีเป็นประธานประชุมครม."]))

ทดสอบ ROUGE: {'rouge1': 80.0, 'rouge2': 25.0, 'rougeL': 60.0}


## 4. การตัดประโยคและฟังก์ชัน Lead-k
ใช้ `pythainlp.tokenize.sent_tokenize` (engine `crfcut`) โดยตัดบรรทัดก่อน และรวมประโยคที่สั้นเกินไปเข้าด้วยกัน

In [5]:
import re
from pythainlp.tokenize import sent_tokenize

def split_sentences(body, engine=SENT_ENGINE, min_len=MIN_LEN):
    parts = []
    for line in re.split(r"[\r\n]+", body):
        line = line.strip()
        if line:
            parts += sent_tokenize(line, engine="crfcut") if engine == "crfcut" else line.split()
    sents, buf = [], ""
    for p in parts:
        buf = (buf + " " + p).strip() if buf else p.strip()
        if len(buf) >= min_len:
            sents.append(buf)
            buf = ""
    if buf:
        if sents:
            sents[-1] += " " + buf
        else:
            sents.append(buf)
    return sents

def summarize_lead_one(body, k=3, engine=SENT_ENGINE):
    sents = split_sentences(body, engine=engine)
    return " ".join(sents[:k])

def summarize_lead(bodies, k=3, engine=SENT_ENGINE):
    return [summarize_lead_one(b, k=k, engine=engine) for b in bodies]

## 5. ทดสอบ Lead-1, Lead-2, Lead-3, Lead-4 บน Test Set
เปรียบเทียบคะแนน ROUGE-1, ROUGE-2, ROUGE-L และความยาวตัวอักษรเฉลี่ยของแต่ละค่า k
(ทำการตัดประโยคครั้งเดียวแล้วแคชไว้ เพื่อเพิ่มความเร็วในการคำนวณ)

In [6]:
print(f"กำลังตัดประโยค test set ({len(test_df)} ข่าว) ด้วย engine={SENT_ENGINE}...")
test_sents = [split_sentences(b) for b in test_df.body]

refs = test_df.summary.tolist()
rows = []
lead_preds = {}

for k in K_LIST:
    preds_k = [" ".join(s[:k]) for s in test_sents]
    lead_preds[k] = preds_k
    scores_k = thai_rouge(preds_k, refs)
    avg_char = round(sum(len(p) for p in preds_k) / len(preds_k))
    rows.append({"method": f"Lead-{k}", "k": int(k), **scores_k, "avg_char_len": int(avg_char)})

lead_eval_df = pd.DataFrame(rows)
ref_avg_len = round(test_df.summary.str.len().mean())
print(f"ความยาวเฉลี่ยบทสรุปจริง (Reference): {ref_avg_len} ตัวอักษร\n")
lead_eval_df

กำลังตัดประโยค test set (500 ข่าว) ด้วย engine=crfcut...


ความยาวเฉลี่ยบทสรุปจริง (Reference): 181 ตัวอักษร



,method,k,rouge1,rouge2,rougeL,avg_char_len
0,Lead-1,1,16.10,10.88,14.65,72
1,Lead-2,2,26.06,17.97,23.19,155
2,Lead-3,3,29.31,19.85,25.50,241
3,Lead-4,4,30.39,20.23,25.94,325


## 6. พิสูจน์ข้อสงสัย: ThaiSum ลอกต้นข่าวเยอะจน ROUGE-2 สูงจริงไหม?

เราวิเคราะห์ 3 มิติ:
1. **% Novel 1-grams:** คำในบทสรุปจริงที่ไม่เคยปรากฏในเนื้อข่าวเลย (ถ้าต่ำมาก = ลอกคำเดิม)
2. **% Novel 2-grams:** คู่คำ (bi-grams) ในบทสรุปจริงที่ไม่เคยปรากฏในเนื้อข่าว (ตัวชี้วัดว่าเขียนใหม่หรือลอกเป็นท่อน)
3. **Lead-3 Bi-gram Recall:** คู่คำของบทสรุปจริงที่หยิบมาจาก 3 ประโยคแรกของเนื้อข่าวโดยตรง

In [7]:
def get_ngrams(tokens, n):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

tokenizer = ThaiTokenizer()

def calculate_novelty(body, summary):
    b_toks = tokenizer.tokenize(body)
    s_toks = tokenizer.tokenize(summary)
    
    b_1grams, s_1grams = set(b_toks), set(s_toks)
    novel_1 = (len(s_1grams - b_1grams) / max(len(s_1grams), 1)) * 100
    
    b_2grams = get_ngrams(b_toks, 2)
    s_2grams = get_ngrams(s_toks, 2)
    novel_2 = (len(s_2grams - b_2grams) / max(len(s_2grams), 1)) * 100 if s_2grams else 0.0
    
    return novel_1, novel_2

def lead3_bigram_recall(lead3_text, summary):
    l3_toks = tokenizer.tokenize(lead3_text)
    s_toks = tokenizer.tokenize(summary)
    
    l3_2grams = get_ngrams(l3_toks, 2)
    s_2grams = get_ngrams(s_toks, 2)
    if not s_2grams:
        return 0.0
    return (len(s_2grams & l3_2grams) / len(s_2grams)) * 100

lead3_preds = lead_preds[3]
novelties = [calculate_novelty(b, s) for b, s in zip(test_df.body, test_df.summary)]
l3_recalls = [lead3_bigram_recall(p, s) for p, s in zip(lead3_preds, test_df.summary)]

test_df["novel_1gram_pct"] = [n[0] for n in novelties]
test_df["novel_2gram_pct"] = [n[1] for n in novelties]
test_df["lead3_bigram_recall_pct"] = l3_recalls

print("=== สรุปผลการวิเคราะห์ข้อสงสัย (Copying / Lead Bias Analysis) ===")
print(f"1. Novel 1-grams เฉลี่ยในบทสรุปจริง: {test_df.novel_1gram_pct.mean():.2f}%")
print(f"2. Novel 2-grams เฉลี่ยในบทสรุปจริง: {test_df.novel_2gram_pct.mean():.2f}%")
print(f"3. Bi-grams ของบทสรุปจริงที่หยิบมาจาก Lead-3 โดยตรง (Recall): {test_df.lead3_bigram_recall_pct.mean():.2f}%")

print("\nคำอธิบายผล:")
if test_df.novel_2gram_pct.mean() < 25.0:
    print("-> สมมติฐานถูกต้อง! ค่า Novel 2-grams อยู่ในระดับต่ำมาก แปลว่าบทสรุปจริงใช้คำคู่ตรงกับเนื้อข่าวแทบทั้งหมด")
    print("-> ประกอบกับ Lead-3 Bi-gram Recall สูง แสดงว่าส่วนใหญ่ดึงมาจาก 3 ประโยคแรกของข่าว")
    print("-> นี่เป็นสาเหตุหลักที่ทำให้ Lead-3 ได้คะแนน ROUGE-2 สูงมากเป็นพิเศษ")

=== สรุปผลการวิเคราะห์ข้อสงสัย (Copying / Lead Bias Analysis) ===
1. Novel 1-grams เฉลี่ยในบทสรุปจริง: 20.74%
2. Novel 2-grams เฉลี่ยในบทสรุปจริง: 50.98%
3. Bi-grams ของบทสรุปจริงที่หยิบมาจาก Lead-3 โดยตรง (Recall): 24.83%

คำอธิบายผล:


## 7. เซฟผลลัพธ์และตัวชี้วัด (ส่งต่อให้คนที่ 6 และคนที่ 7)
- บันทึก `predictions_lead3.csv` (มี body, summary, pred, rouge2, rougeL)
- บันทึก `result_lead3.json` เพื่อส่งเป็นหลักฐานให้คนที่ 7 รวมผลและเขียนรายงาน

In [8]:
import json

lead3_scores = thai_rouge(lead3_preds, refs)

out_df = test_df.copy()
out_df["pred"] = lead3_preds

# คำนวณ ROUGE ทีละข่าวรอบเดียวเพื่อประสิทธิภาพ
sample_scores = [scorer.score(r, p) for p, r in zip(lead3_preds, refs)]
out_df["rouge1"] = [s["rouge1"].fmeasure for s in sample_scores]
out_df["rouge2"] = [s["rouge2"].fmeasure for s in sample_scores]
out_df["rougeL"] = [s["rougeL"].fmeasure for s in sample_scores]

# เซฟไฟล์ CSV
out_df.to_csv(f"predictions_{OUTPUT_NAME}.csv", index=False)
print(f"บันทึกไฟล์ predictions_{OUTPUT_NAME}.csv เรียบร้อย ({len(out_df)} แถว)")

# เซฟไฟล์ JSON (ใช้ rows ที่เป็น native Python int/float เพื่อป้องกัน TypeError จาก numpy.int64)
summary_result = {
    "model": "Lead-3 (Baseline)",
    **lead3_scores,
    "lead_k_comparison": rows,
    "copying_analysis": {
        "novel_1gram_pct_mean": round(float(test_df.novel_1gram_pct.mean()), 2),
        "novel_2gram_pct_mean": round(float(test_df.novel_2gram_pct.mean()), 2),
        "lead3_bigram_recall_pct_mean": round(float(test_df.lead3_bigram_recall_pct.mean()), 2),
        "body_avg_chars": round(float(test_df.body.str.len().mean()), 1),
        "ref_summary_avg_chars": round(float(test_df.summary.str.len().mean()), 1),
        "lead3_pred_avg_chars": round(float(out_df.pred.str.len().mean()), 1),
    },
    "test_size": len(test_df),
    "sent_engine": SENT_ENGINE
}

with open(f"result_{OUTPUT_NAME}.json", "w", encoding="utf-8") as f:
    json.dump(summary_result, f, ensure_ascii=False, indent=2)

print(json.dumps(summary_result, ensure_ascii=False, indent=2))

บันทึกไฟล์ predictions_lead3.csv เรียบร้อย (500 แถว)
{
  "model": "Lead-3 (Baseline)",
  "rouge1": 29.31,
  "rouge2": 19.85,
  "rougeL": 25.5,
  "lead_k_comparison": [
    {
      "method": "Lead-1",
      "k": 1,
      "rouge1": 16.1,
      "rouge2": 10.88,
      "rougeL": 14.65,
      "avg_char_len": 72
    },
    {
      "method": "Lead-2",
      "k": 2,
      "rouge1": 26.06,
      "rouge2": 17.97,
      "rougeL": 23.19,
      "avg_char_len": 155
    },
    {
      "method": "Lead-3",
      "k": 3,
      "rouge1": 29.31,
      "rouge2": 19.85,
      "rougeL": 25.5,
      "avg_char_len": 241
    },
    {
      "method": "Lead-4",
      "k": 4,
      "rouge1": 30.39,
      "rouge2": 20.23,
      "rougeL": 25.94,
      "avg_char_len": 325
    }
  ],
  "copying_analysis": {
    "novel_1gram_pct_mean": 20.74,
    "novel_2gram_pct_mean": 50.98,
    "lead3_bigram_recall_pct_mean": 24.83,
    "body_avg_chars": 2559.8,
    "ref_summary_avg_chars": 180.8,
    "lead3_pred_avg_chars": 240.6
  

## 8. ส่องตัวอย่างจริง: ดูข่าวที่ Lead-3 ได้ ROUGE-2 สูงสุด และ ต่ำสุด

In [9]:
from IPython.display import display

pd.set_option("display.max_colwidth", 200)
print("=== 3 ข่าวที่ Lead-3 ได้ ROUGE-2 สูงสุด (ลอก 3 ประโยคแรกชัดเจน) ===")
display(out_df.sort_values("rouge2", ascending=False)[["summary", "pred", "rouge2", "lead3_bigram_recall_pct"]].head(3))

print("\n=== 3 ข่าวที่ Lead-3 ได้ ROUGE-2 ต่ำสุด (เขียนใหม่ หรือสรุปจากท้ายข่าว) ===")
display(out_df.sort_values("rouge2", ascending=True)[["summary", "pred", "rouge2", "novel_2gram_pct"]].head(3))

=== 3 ข่าวที่ Lead-3 ได้ ROUGE-2 สูงสุด (ลอก 3 ประโยคแรกชัดเจน) ===


,summary,pred,rouge2,lead3_bigram_recall_pct
184,ทนาย 11 นักศึกษาที่ โดนหมายเรียกกรณีขึ้น รถไฟไปอุทยานราชภักดิ์ ยื่นหนังสือขอเลื่อน พบพนักงานสอบสวน เพื่อรับทราบข้อกล่าวหา ไปเป็นวันที่ 9 ม.ค.ปีหน้า อ้างกระชั้นชิดเกินไป,ทนาย 11 นักศึกษาที่ โดนหมายเรียกกรณีขึ้นรถไฟไปอุทยานราชภักดิ์ ยื่นหนังสือขอเลื่อน พบพนักงานสอบสวน เพื่อรับทราบข้อกล่าวหา ไปเป็นวันที่ 9 ม.ค. ปีหน้า อ้างกระชั้นชิดเกินไป,1.000000,100.0
38,กปน. เปิดลงทะเบียนออนไลน์ขอคืนเงินประกันการใช้น้ำ โดยสามารถตรวจสอบสิทธิ์ ผ่านเว็บ ไลน์ แอปฯ ตั้งแต่ 1 พ.ค.63 พร้อมเร่งคืนเงินตั้งแต่ 15 พ.ค.63,กปน. เปิดลงทะเบียนออนไลน์ขอคืนเงินประกันการใช้น้ำ โดยสามารถตรวจสอบสิทธิ์ ผ่านเว็บ ไลน์ แอปฯ ตั้งแต่ 1 พ.ค. 63 พร้อมเร่งคืนเงินตั้งแต่ 15 พ.ค.,0.982456,100.0
318,เจอสายหื่นทักเป็นเรื่องจนได้ หลังราชินีแดนซ์คนล่าสุดของเมืองไทย กระแต อาร์สยาม หรือ แตร บุญยะเลี้ยง เจ้าของผลงานเพลง สาวดอยคอยปี้ กระหน่ำโพสต์ภาพลงโซเชียลส่วนตัวทั้งเพจเฟซบุ๊ก,เจอสายหื่นทักเป็นเรื่องจนได้ หลังราชินีแดนซ์คนล่าสุดของเมืองไทย กระแต อาร์สยาม หรือ แตร บุญยะเลี้ยง เจ้าของผลงานเพลงสาวดอยคอยปี้ กระหน่ำโพสต์ภาพลงโซเชียลส่วนตัวทั้งเพจเฟซบุ๊ก @กระแต,0.973684,100.0



=== 3 ข่าวที่ Lead-3 ได้ ROUGE-2 ต่ำสุด (เขียนใหม่ หรือสรุปจากท้ายข่าว) ===


,summary,pred,rouge2,novel_2gram_pct
15,โปรดเกล้าฯ พ.ร.บ.คณะสงฆ์ ให้พระมหากษัตริย์ทรงแต่งตั้งกรรมการมหาเถรสมาคมจากพระภิกษุ ซึ่งมีพรรษาอันสมควร และมีจริยวัตรเหมาะสมแก่การปกครองคณะสงฆ์,เมื่อวันที่ 18 ก.ค. 61 ผู้สื่อข่าวรายงานว่า วานนี้(17 ก.ค. ) เว็บไซต์ราชกิจจานุเบกษา ได้เผยแพร่พระราชบัญญัติ(พ.ร.บ.,0.0,20.833333
18,กองปราบร่วมลุย บุกค้นบ้านเสี่ยตงที่เมืองกาญจน์ อายัดเสือโคร่งพันธุ์เบงกอล 4 ตัว เชื่อเชื่อมโยงวัดเสือ หรืออาจเป็นที่ชำแหละ ด้าน ส.ป.ก.ติดป้ายยึดที่วัดป่าหลวงตาบัวฯ เตรียมใช้ที่ตั้งศูนย์เรียนรู้ตาม...,เมื่อเวลา 10.00 น. วันที่ 7 มิ.ย. 59 พ.ต.อ. มนตรี แป้นเจริญ รอง,0.0,69.047619
465,หนุ่มใหญ่ ควบกระบะกลับบ้าน ชน จยย.พ่วงข้างขับสวนทาง ลากถูพื้นไกลกว่า 30 เมตร ทำคนขับ จยย.เสียชีวิต คนขับกระบะอ้างทางแคบ-มองไม่เห็น,เมื่อเวลา 20.20 น. วันที่ 3 พ.ค. 63 ร.ต.อ. ชาติศิริ จี๋คำวัน รอง สว.,0.0,92.000000
